# From Hodgkin–Huxley to a channelopathy: writing your own neuron in NESTML

**Goals**
1. Explore the classic Hodgkin–Huxley (HH) neuron built into NEST.
2. Re-create the same neuron in NESTML, generate code for it, and check that it behaves identically.
3. **Edit the NESTML file** to add a disease-causing sodium-channel mutation, regenerate, and compare wild type vs. mutant.

**Kernel:** on EBRAINS JupyterLab choose the most recent *EBRAINS-xx.xx* kernel (it ships NEST + NESTML).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import nest
import pynestml
from pynestml.codegeneration.nest_code_generator_utils import NESTCodeGeneratorUtils

print("NEST  :", nest.__version__)
print("NESTML:", pynestml.__version__)

---
# Part 1 – Play with the built-in HH neuron (`hh_psc_alpha`)

In [ ]:
plt.figure(figsize=(20, 4))
for amp in [300., 700., 1500.]:

    nest.ResetKernel()
    nest.set_verbosity("M_WARNING")
    
    n = nest.Create("hh_psc_alpha")

    stim = nest.Create("step_current_generator",
                       params={"amplitude_times": [50., 250.], "amplitude_values": [amp, 0.]})
    vm = nest.Create("voltmeter", params={"interval": 0.1})
    sr = nest.Create("spike_recorder")

    nest.Connect(stim, n)
    nest.Connect(vm, n)
    nest.Connect(n, sr)

    nest.Simulate(300.0)

    t = vm.get("events")["times"]
    V = vm.get("events")["V_m"]
    spk = sr.get("events")["times"]

    plt.plot(t, V, label=f"{amp:.0f} pA ({len(spk)} spikes)")

plt.xlabel("time (ms)")
plt.ylabel("V_m (mV)")
plt.legend()
plt.title("hh_psc_alpha, current steps 50–250 ms")
plt.show()

In [ ]:
currents = np.arange(0., 2001., 50.)
t_sim, t_discard = 1000., 200.

nest.ResetKernel()
nest.set_verbosity("M_WARNING")

pop = nest.Create("hh_psc_alpha", len(currents))
pop.I_e = list(currents)

sr = nest.Create("spike_recorder")

nest.Connect(pop, sr)

nest.Simulate(t_sim)

senders = sr.get("events")["senders"]
times = sr.get("events")["times"]
rate_builtin = [np.sum((senders == gid) & (times > t_discard)) / ((t_sim - t_discard) / 1000.)
                for gid in pop.tolist()]

plt.figure(figsize=(10, 4))
plt.plot(currents, rate_builtin, "o-")
plt.xlabel("I_e (pA)")
plt.ylabel("rate (Hz)")
plt.title("f–I curve, built-in HH")
plt.show()

**Questions (Part 1)**
- What is the minimal current that produces repetitive firing (rheobase)? Is the onset of firing continuous (type I) or abrupt (type II)?
- Change `g_K` or `g_Na` via `params={...}` and describe what happens.
- Repeat the f–I curve with `t_discard = 0`. Where does the curve change, and why?

---
# Part 2 – The same neuron written in NESTML

The next cell writes the model to the file **`hh_psc_alpha_neuron.nestml`** (you'll see it appear in the file browser on the left). Open the NESTML file and read it carefully: the `equations` block contains exactly the HH equations you know from the lecture.

In [ ]:
# Generate C++ code, compile it and load it into NEST (takes ~1 minute)

wt_module, wt_model = NESTCodeGeneratorUtils.generate_code_for(open("hh_psc_alpha_neuron.nestml").read())

**Sanity check:** the NESTML neuron must behave like the built-in one.

In [ ]:
amp = 700.

plt.figure(figsize=(20, 4))

nest.ResetKernel()
nest.set_verbosity("M_WARNING")
nest.Install(wt_module)

n = nest.Create("hh_psc_alpha")
wt = nest.Create(wt_model)

stim = nest.Create("step_current_generator",
                   params={"amplitude_times": [50., 250.], "amplitude_values": [amp, 0.]})
vm = nest.Create("voltmeter", params={"interval": 0.1})
sr = nest.Create("spike_recorder")

nest.Connect(stim, n); nest.Connect(stim, wt)
nest.Connect(vm, n); nest.Connect(vm, wt)
nest.Connect(n, sr); nest.Connect(wt, sr)

nest.Simulate(300.0)

ev = vm.get("events")
spk = sr.get("events")

# the voltmeter records both neurons: separate them by sender id
for neuron, name, style in [(n, "Built-in HH", "-"), (wt, "NESTML HH (wt)", "--")]:
    gid = neuron.global_id
    sel = ev["senders"] == gid
    n_spk = sum(spk["senders"] == gid)
    plt.plot(ev["times"][sel], ev["V_m"][sel], style, label=f"{name}, {amp:.0f} pA ({n_spk} spikes)")
    
plt.xlabel("time (ms)")
plt.ylabel("V_m (mV)")
plt.legend()
plt.title("Built-in vs. NESTML HH, current step 50–250 ms")
plt.show()

---
# Part 3 – Your task: model a sodium channelopathy

Many inherited diseases (some epilepsies, paramyotonia congenita, long-QT syndrome type 3, hyperkalemic periodic paralysis) are caused by mutations in voltage-gated **Na⁺ channels** that impair **fast inactivation**: the mutant channel opens normally, but does not close again.

**Extremely simplified model:** a fraction `p_mut` of all Na⁺ channels is mutant and has *no h gate*:

$$I_{Na,WT} = (1-p_{mut})\,\bar g_{Na}\, m^3 h\,(V-E_{Na}) \qquad I_{Na,mut} = p_{mut}\,\bar g_{Na}\, m^3\,(V-E_{Na})$$

Run the next cell: it creates **`hh_mut_neuron.nestml`**, a copy of the wild-type file. Then **open `hh_mut_neuron.nestml` in the JupyterLab editor** (double-click it in the file browser) and:

1. Add a new parameter `p_mut real = 0.02` in the `parameters` block.
2. Multiply the existing `I_Na` by `(1 - p_mut)`.
3. Add a new inline equation for the mutant channels (it has no `Act_h`!): `inline I_NaMut pA = p_mut * g_Na * Act_m**3 * (V_m - E_Na)`
4. Add `I_NaMut` to the membrane equation `V_m'`.
5. Save the file (Ctrl+S).

*Hint:* units must be consistent – `I_NaMut` is in `pA`, like `I_Na`. NESTML will tell you if they are not.

In [ ]:
# After editing and saving the file: generate and load the mutant model

mut_module, mut_model = NESTCodeGeneratorUtils.generate_code_for(open("hh_psc_alpha_neuron_mut.nestml").read())

---
# Part 4 – Wild type vs. mutant

Because `p_mut` is a *parameter*, we can change it from Python without regenerating code.

**Check 1:** with `p_mut = 0` the mutant model must be identical to the wild type. Why?

In [ ]:
amp = 600.

plt.figure(figsize=(20, 4))


for module, model, params, name in [(wt_module, wt_model,  {},              "wild type"),
                                    (mut_module, mut_model, {"p_mut": 0.0},  "p_mut = 0"),
                                    (mut_module, mut_model, {"p_mut": 0.05}, "p_mut = 0.05"),
                                    (mut_module, mut_model, {"p_mut": 0.2},  "p_mut = 0.2")]:
    
    nest.ResetKernel()
    nest.set_verbosity("M_WARNING")
    nest.Install(module)
    
    n = nest.Create(model, params=params)

    stim = nest.Create("step_current_generator",
                       params={"amplitude_times": [50., 250.], "amplitude_values": [amp, 0.]})
    vm = nest.Create("voltmeter", params={"interval": 0.1})
    sr = nest.Create("spike_recorder")

    nest.Connect(stim, n)
    nest.Connect(vm, n)
    nest.Connect(n, sr)

    nest.Simulate(300.0)

    t = vm.get("events")["times"]
    V = vm.get("events")["V_m"]
    spk = sr.get("events")["times"]

    plt.plot(t, V, label=f"{name} ({len(spk)} spikes)")

plt.xlabel("time (ms)")
plt.ylabel("V_m (mV)")
plt.legend()
plt.title(f"Wild type vs. mutant, current step {amp:.0f} pA, 50–250 ms")
plt.show()

In [ ]:
currents = np.arange(0., 2001., 50.)
t_sim, t_discard = 1000., 200.

plt.figure(figsize=(10, 4))

for module, model, params, name in [(wt_module,  wt_model,  {},               "wild type"),
                                    (mut_module, mut_model, {"p_mut": 0.02},  "p_mut = 0.02"),
                                    (mut_module, mut_model, {"p_mut": 0.05},  "p_mut = 0.05"),
                                    (mut_module, mut_model, {"p_mut": 0.12},  "p_mut = 0.12"),
                                    (mut_module, mut_model, {"p_mut": 0.2},   "p_mut = 0.2")]:
    
    nest.ResetKernel()
    nest.set_verbosity("M_WARNING")
    nest.Install(module)

    pop = nest.Create(model, len(currents), params=params)
    pop.I_e = list(currents)

    sr = nest.Create("spike_recorder")

    nest.Connect(pop, sr)

    nest.Simulate(t_sim)

    senders = sr.get("events")["senders"]
    times = sr.get("events")["times"]
    rates = [np.sum((senders == gid) & (times > t_discard)) / ((t_sim - t_discard) / 1000.)
             for gid in pop.tolist()]

    plt.plot(currents, rates, "o-", ms=3, label=name)

plt.xlabel("I_e (pA)")
plt.ylabel("rate (Hz)")
plt.legend()
plt.title("f–I curves")
plt.show()